# FI-1 — Face Intelligence

Face Intelligence provides four detection capabilities in a single model. You can enable any combination in one call.

**Model:** `FI-1`  
**Input:** Image (`.jpg`, `.jpeg`, `.png`) or Video (`.mp4`, `.mov`)

| Capability | Parameter | Modality |
| :-- | :-- | :-- |
| Liveness Detection | `livenessCheck=True` | Image / Video |
| Face Swap Detection | `faceswapCheck=True` | **Video only** |
| Face Similarity | `faceSimilarityCheck=True` | **Image only** |
| Single Face Validation | `isSingleFace=True` | Image / Video |

**How results work:**
- `face_intelligence(auto_polling=True)` returns a media dict with a `result` key  
  containing the full detection output fetched automatically from `resultURL`.
- You can also call `client.get_result(media)` manually on any processed media dict.

---
### Contents
1. Setup
2. Liveness Detection — video
3. Liveness Detection — image
4. Face Swap Detection
5. Face Similarity Check
6. `get_result()` — fetch result from any processed media
7. Manual Polling (`auto_polling=False`)
8. Error handling
9. Async client

---
## 1. Setup

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath('../src'))

from authenta.authenta_client import AuthentaClient
from authenta import (
    AuthentaError,
    AuthenticationError,
    QuotaExceededError,
    InsufficientCreditsError,
)

CLIENT_ID     = "YOUR_CLIENT_ID"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"
BASE_URL      = "https://platform.authenta.ai"

client = AuthentaClient(
    base_url=BASE_URL,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

print("Client ready.")

---
## 2. Liveness Detection — Video

Determine whether the face in a video is live (real) or a presentation attack.

When `auto_polling=True`, the SDK automatically fetches the detection output from `resultURL`  
and attaches it to the returned dict under `media['result']`.

In [ ]:
# Real face video
media = client.face_intelligence(
    path="../data_samples/face_live_video/real/1.mp4",
    model_type="FI-1",
    livenessCheck=True,
    auto_polling=True,
)

print(f"Media ID : {media['mid']}")
print(f"Status   : {media['status']}")
print(f"Result   :")
print(json.dumps(media['result'], indent=2))

In [ ]:
# Fake (presentation attack) video
media = client.face_intelligence(
    path="../data_samples/face_live_video/fake/1.mp4",
    model_type="FI-1",
    livenessCheck=True,
    auto_polling=True,
)

print(f"Status : {media['status']}")
print(f"Result :")
print(json.dumps(media['result'], indent=2))

---
## 3. Liveness Detection — Image

In [ ]:
# Real face image
media = client.face_intelligence(
    path="../data_samples/face_live_images/real/1.jpg",
    model_type="FI-1",
    livenessCheck=True,
    auto_polling=True,
)

print(f"Status : {media['status']}")
print(f"Result :")
print(json.dumps(media['result'], indent=2))

In [ ]:
# Fake face image
media = client.face_intelligence(
    path="../data_samples/face_live_images/fake/1.png",
    model_type="FI-1",
    livenessCheck=True,
    auto_polling=True,
)

print(f"Status : {media['status']}")
print(f"Result :")
print(json.dumps(media['result'], indent=2))

---
## 4. Face Swap Detection

Detect whether a face in a video has been swapped using AI. **Video only.**

In [ ]:
# Fake (swapped) video
media = client.face_intelligence(
    path="../data_samples/faceswap/fake/1.mp4",
    model_type="FI-1",
    faceswapCheck=True,
    auto_polling=True,
)

print(f"Media ID : {media['mid']}")
print(f"Status   : {media['status']}")
print(f"Result   :")
print(json.dumps(media['result'], indent=2))

In [ ]:
# Real (no swap) video
media = client.face_intelligence(
    path="../data_samples/faceswap/real/1.mp4",
    model_type="FI-1",
    faceswapCheck=True,
    auto_polling=True,
)

print(f"Status : {media['status']}")
print(f"Result :")
print(json.dumps(media['result'], indent=2))

---
## 5. Face Similarity Check

Compare two face images and determine whether they belong to the same person. **Image only.**  
`reference_img_path` is required.

In [ ]:
# Same person
media = client.face_intelligence(
    path="../data_samples/face_similiar/person_1/A.jpeg",
    reference_img_path="../data_samples/face_similiar/person_1/B.jpeg",
    model_type="FI-1",
    faceSimilarityCheck=True,
    auto_polling=True,
)

print(f"Media ID : {media['mid']}")
print(f"Status   : {media['status']}")
print(f"Result   :")
print(json.dumps(media['result'], indent=2))

In [ ]:
# Different persons
media = client.face_intelligence(
    path="../data_samples/face_similiar/person_1/A.jpeg",
    reference_img_path="../data_samples/face_similiar/person_2/A.jpeg",
    model_type="FI-1",
    faceSimilarityCheck=True,
    auto_polling=True,
)

print(f"Status : {media['status']}")
print(f"Result :")
print(json.dumps(media['result'], indent=2))

---
## 6. `get_result()` — Fetch Result from Any Processed Media

`client.get_result(media)` fetches the detection output from `resultURL` for any processed media dict.  
Use this when you have a media object from `get_media()`, `wait_for_media()`, or when you stored  
a `mid` and retrieved it later.

In [ ]:
# Upload without auto_polling, then poll manually, then fetch result explicitly
upload_meta = client.face_intelligence(
    path="../data_samples/face_live_video/real/1.mp4",
    model_type="FI-1",
    livenessCheck=True,
    auto_polling=False,
)
mid = upload_meta["mid"]
print(f"Uploaded: {mid}")

# Poll until done
media = client.wait_for_media(mid)
print(f"Status: {media['status']}")

# Fetch result explicitly
result = client.get_result(media)
print(f"Result:")
print(json.dumps(result, indent=2))

In [ ]:
# Or fetch result from a previously known mid
KNOWN_MID = "YOUR_MEDIA_ID"   # replace with a real mid

media = client.get_media(KNOWN_MID)
if media["status"] == "PROCESSED":
    result = client.get_result(media)
    print(json.dumps(result, indent=2))
else:
    print(f"Not ready yet: {media['status']}")

---
## 7. Manual Polling (`auto_polling=False`)

Return immediately after upload, then poll manually and fetch result when ready.

In [ ]:
# Step 1 — upload without blocking
upload_meta = client.face_intelligence(
    path="../data_samples/face_live_video/real/1.mp4",
    model_type="FI-1",
    livenessCheck=True,
    auto_polling=False,
)

mid = upload_meta["mid"]
print(f"Upload started. Media ID: {mid}")
print(f"Status: {upload_meta['status']}")

In [ ]:
# Step 2 — poll when ready
media = client.wait_for_media(mid, interval=5.0, timeout=600.0)
print(f"Status: {media['status']}")

# Step 3 — fetch result
result = client.get_result(media)
print(f"Result:")
print(json.dumps(result, indent=2))

---
## 8. Error Handling

### 8a. Invalid parameter combinations

In [ ]:
# faceswapCheck on an image → ValueError
try:
    client.face_intelligence(
        path="../data_samples/face_similiar/person_1/A.jpeg",
        model_type="FI-1",
        faceswapCheck=True,
    )
except ValueError as e:
    print(f"Caught: {e}")

In [ ]:
# faceSimilarityCheck on a video → ValueError
try:
    client.face_intelligence(
        path="../data_samples/face_live_video/real/1.mp4",
        model_type="FI-1",
        faceSimilarityCheck=True,
    )
except ValueError as e:
    print(f"Caught: {e}")

In [ ]:
# faceSimilarityCheck without reference_img_path → ValueError
try:
    client.face_intelligence(
        path="../data_samples/face_similiar/person_1/A.jpeg",
        model_type="FI-1",
        faceSimilarityCheck=True,
    )
except ValueError as e:
    print(f"Caught: {e}")

### 8b. API errors

In [ ]:
try:
    media = client.face_intelligence(
        path="../data_samples/face_live_video/real/1.mp4",
        model_type="FI-1",
        livenessCheck=True,
        auto_polling=True,
    )
    print(json.dumps(media['result'], indent=2))
except AuthenticationError:
    print("Authentication failed — check your CLIENT_ID and CLIENT_SECRET.")
except QuotaExceededError:
    print("API quota exceeded — upgrade your plan.")
except InsufficientCreditsError:
    print("Not enough credits.")
except TimeoutError as e:
    print(f"Timed out: {e}")
except AuthentaError as e:
    print(f"API error [{e.code}]: {e.message}")

---
## 9. Async Client

The async client's `process_FI()` also supports `auto_polling=True` which returns `media['result']`  
automatically. For manual polling, use `await client.wait_for_media(mid)` then  
`client.get_result(media)` (sync — no await needed as it's a plain HTTP GET).

In [ ]:
import asyncio
from authenta.async_authenta_client import AsyncAuthentaClient

# Liveness — async
async def liveness_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as async_client:
        media = await async_client.process_FI(
            path="../data_samples/face_live_video/real/1.mp4",
            model_type="FI-1",
            livenessCheck=True,
            auto_polling=True,
        )
        print(f"Status : {media['status']}")
        # Fetch result using the sync client helper
        result = client.get_result(media)
        print(f"Result :")
        print(json.dumps(result, indent=2))

await liveness_async()

In [ ]:
# Face swap — async
async def faceswap_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as async_client:
        media = await async_client.process_FI(
            path="../data_samples/faceswap/fake/1.mp4",
            model_type="FI-1",
            faceSwapCheck=True,
            auto_polling=True,
        )
        result = client.get_result(media)
        print(f"Status : {media['status']}")
        print(json.dumps(result, indent=2))

await faceswap_async()

In [ ]:
# Face similarity — async
async def similarity_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as async_client:
        media = await async_client.process_FI(
            path="../data_samples/face_similiar/person_1/A.jpeg",
            model_type="FI-1",
            faceSimilarityCheck=True,
            auto_polling=True,
        )
        result = client.get_result(media)
        print(f"Status : {media['status']}")
        print(json.dumps(result, indent=2))

await similarity_async()

In [ ]:
# Manual polling — async, then get_result
async def manual_poll_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as async_client:
        # Step 1 — upload without blocking
        upload_meta = await async_client.process_FI(
            path="../data_samples/face_live_video/real/1.mp4",
            model_type="FI-1",
            livenessCheck=True,
            auto_polling=False,
        )
        mid = upload_meta["mid"]
        print(f"Uploaded. mid={mid}")

        # Step 2 — poll when ready
        media = await async_client.wait_for_media(mid)
        print(f"Status: {media['status']}")

        # Step 3 — fetch result
        result = client.get_result(media)
        print(json.dumps(result, indent=2))

await manual_poll_async()